In [ ]:
# First setup context again:
import strax
import straxen 
import cutax
straxen.print_versions()

# Some other packages
import numpy as np
import matplotlib.pyplot as plt
import os
import getpass
user = getpass.getuser() #Figure out who is running this in order to set paths.

Initialize the same context as in the first two tutorials

In [ ]:
st = cutax.contexts.xenonnt_online(output_folder=f'/scratch/midway3/{user}/strax_data')

# Strax Plugin, "An information transformer machine":

From tutorial 1, you have already seen what plugins are and how they interact with each other in this notebook we want to go a bit deeper into the technicalities and focus on the question: "How can I develop my own plugins?". 

However, before we go into detail lets recap the most important facts you need to know about the data structure in strax:

* The data of a strax based framework is organized in chunks and arrays. Each chunk represents a disjoint time interval containing the data as a [numpy.structure.array](https://numpy.org/doc/stable/user/basics.rec.html).
* The data (numpy.structure.array) can be imagined as table each row represents a time interval. These time intervals are represented either by a `time` and `endtime` field (column) or a `time`, `dt` and `length` field. 
* The data within a chunk is sorted first by time and then by channel. In your plugin you must ensure that this order is kept in the end. 
* Zero duration rows are not allowed.

To organize data in such a scheme has the advantage that data can be processed in parallel loops as the information in a chunk is independent from a another chunk. 

In the subsequent sketch we show as an example the symbolic data processing from the plugin "PulseProcessing" to "PeakProcessing". Each of the red and blue boxes represents a single chunk. The assembly belt represent the different parallel processes delivering the data. In the bottom right the interior numpy.sturcture.array of a chunk is shown. Since, numpy can only handled fixed length objects, things like the raw pulses coming from our digitzers have to be split across multiple rows.  


<center><img src="images/StraxChunks.png" width="1200"></center>

## What are the key ingredients of a Plugin?

### A Plugin Must Provide:
- A data type that is made
- A way of computing the data type
- Necessary configuration options to set up the computation of the data-type
- Additional functions to speed up the processing

### Plugin types:

* strax.Plugin: This is the default plugin type used in straxen. It can depend on one or more data_kinds.
* strax.OverlapWindowPlugin: An OverlapWindow plugin allows to look for data in adjacent chunks. This only works for disjoint intervals like peaks. 
* strax.LoopPlugin (legacy): A LoopPlugin takes by default two data_kinds as input. It loops over the first "base" data_kind and checks which of the intervals in the second data_kind overlap with it. This plugin type should only be used for high level data as it loops over the base data_kind via a for loop. If you need to use it on low level data you should fall back to a normal strax.Plugin and use other means. Just let us know and we can explain you how.
* strax.CutPlugin: A plugin which returns a boolean array that can be used to select or cut data. This return type of a CutPlugin, if registered to the context, can be used in `get_array(run_id, target, selection_str='cut_plugin_provide')`.

For more information and additional plugin types, please refer to the [strax documentation](https://strax.readthedocs.io/en/latest/advanced/plugin_dev.html).

<div style="    
    color: black;
    background-color: #F6D092;
    border-style: solid;
    border-color: darkorange;
    border-radius : 5pt; 
    padding-left: 5pt" markdown="1">
    
### Exercise 1. Plugin types: 

Think about the XENONnT data processing from pulses to events. At which stage do we need which type of plugin?

<details markdown="1">
<summary markdown="1">

**Solution (click me):**

</summary>

* strax.Plugin are mostly used in our analysis framework. There are always suited if we want to compute certain properties of a data_kind which only depend on the data_kind itself. E.g. straxen.PulseProcessing or straxen.PeakBasics.
* strax.OverlapWindowPlugin are usually needed if you build from one data_kind the next higher data_kind. E.g. straxen.MergedS2s, straxen.Events.
* strax.LoopPlugin are helpful if you want to compute the properties of one data_kind which depend on the properties of another data_kind. E.g. straxen.EventBasics. 

</details>
</div>

</div>

## Example: nVETOHitlets
Below you can find the full code of a typical default plugin.

Usually consists of:
- A doc string to say what the plugin does
- `depends_on`: the data types that the plugin takes as input
- `provides`: the output of the plugin, used in the `targets` argument in `get_array`
- Either `dtype` or an `infer_dtype()` method that provides the dtype for the structured array output
- `compute(self, data_kind1, data_kind2, ...)`
- Functions that are called in `compute` to optimize the code (often times using `numba`)

<center><img src="images/HitletsNVPluginExample.png" width="800"/></center>

In the following lines we will go through the different aspects of a plugin and explain them one by one.

## Plugin Type, Header, and dtype

```python
class nVETOHitlets(strax.Plugin):
    """
    Plugin which computes the nveto hitlets and their parameters.

    Hitlets are an extension of regular hits. They include the left
    and right extension. The plugin does the following:
        1. Generate hitlets which includes these sub-steps:
            * Apply left and right hit extension and concatenate
            overlapping hits.
            * Generate temp. hitelts and look for their waveforms in
            their corresponding records.
            * Split hitlets if they satisfy the set criteria.
        2. Compute the properties of the hitlets.

    Note:
        Hitlets are getting chopped if extended in not recorded regions.
    """
    __version__ = '0.1.1'

    parallel = 'process'
    rechunk_on_save = True
    compressor = 'zstd'

    depends_on = 'records_nv'

    provides = 'hitlets_nv'
    data_kind = 'hitlets_nv'
```

The class itself is inherited from one of the supported strax.Plugin types. In this example it is the base strax.Plugin class. Each Plugin requires either a tuple or a string of data_types the plugin `depends_on`. The same holds true for its return which is indicated by `provides`. By default the new `data_type` returned by a plugin has the same data_kind as the plugin input. If this is not desired one has to change the `data_kind` into a new data_kind.

In order to be able to track changes in the code each plugin has its own `__version__` and whenever you make a change in the code which requires a reprocessing of the data you should increase the version. <span style="color: red;">
    *An instance where this matters*: When developing a plugin, the `__version__` is used in determining the lineage hash. As such, if the version is not incremented upon changing the plugin, the output may not reflect those changes, and calling `get_array` may just return the old version. 
</span>During development you also can specify the version as None to avoid to increment the version every time you make a little change.

Besides these mandatory fields there are a bunch of optional settings like the `parallel=True` flag. The flag indicates that we can process several `hitlets_nv` chunks in parallel. To get an overview over all possible option you can check the [strax code](https://github.com/AxFoundation/strax/blob/58544a7d829178019b78dbcd31d4bc1626eda382/strax/plugin.py#L41-L99) directly, but better ask first in [slack](https://xenonnt.slack.com/archives/C016UNNB5DZ) before trying things out.  

For multiple input data types:
 - `depends_on` can be specified as a tuple of different data types such as `depends_on = ('event_basics', 'peak_basics', 'peak_positions')`
 - The `compute` method will then need the arguments `(events, peaks)`
 - Each argument represents a data *kind*

For multiple outputs:
 - The return must be a dictionary of the form `{'type' : data_of_type}`

To see the differences for a plugin with [multiple dependencies](https://github.com/XENONnT/straxen/blob/e0bf0f21c797ec0008bc13b779d6c00a6ae4fcbb/straxen/plugins/peaklet_processing.py#L510) or [multiple provides](https://github.com/XENONnT/straxen/blob/e0bf0f21c797ec0008bc13b779d6c00a6ae4fcbb/straxen/plugins/pulse_processing.py#L97-L142) you can check the linked plugins.

## URL Configs
<span style="color:red">Plugin config options used to be specified with the decorator `@strax.takes_config` with a default value, a type, and a help string. However, this meant that non-trivial tasks such as loading files, using a tensorflow model, or looking up values from the corrections management tool (CMT) must be manually implemented in a `setup()` method. Now, with URL configs, we can load these things in a URL-like format. More information is provided in the straxen documentation on [URL Configs](https://straxen.readthedocs.io/en/latest/url_configs.html).</span>

```python
    save_outside_hits_nv = straxen.URLConfig(
        default=NV_HIT_DEFAULTS['save_outside_hits_nv'], track=True, infer_type=False,
        help='Save (left, right) samples besides hits; cut the rest')

    hit_min_amplitude_nv = straxen.URLConfig(
        infer_type=False,
        default=NV_HIT_DEFAULTS['hit_min_amplitude_nv'], track=True,
        help='Minimum hit amplitude in ADC counts above baseline. '
             'Specify as a tuple of length n_nveto_pmts, or a number, '
             'or a string like "pmt_commissioning_initial" which means calling '
             'hitfinder_thresholds.py, '
             'or a tuple like (correction=str, version=str, nT=boolean), '
             'which means we are using cmt.')

    min_split_nv = straxen.URLConfig(
        default=0.063, track=True, infer_type=False,
        help='Minimum height difference pe/sample between local minimum and maximum, '
             'that a pulse get split.')

    min_split_ratio_nv = straxen.URLConfig(
        default=0.75, track=True, infer_type=False,
        help='Min ratio between local maximum and minimum to split pulse (zero to switch this '
             'off).')

    entropy_template_nv = straxen.URLConfig(
        default='flat', track=True, infer_type=False,
        help='Template data is compared with in conditional entropy. Can be either "flat" or an '
             'template array.')

    entropy_square_data_nv = straxen.URLConfig(
        default=False, track=True, infer_type=False,
        help='Parameter which decides if data is first squared before normalized and compared to '
             'the template.')

    channel_map = straxen.URLConfig(track=False, type=immutabledict,
                                    help="immutabledict mapping subdetector to (min, max) "
                                         "channel number.")

    gain_model_nv = straxen.URLConfig(
        default="cmt://to_pe_model_nv?version=ONLINE&run_id=plugin.run_id", infer_type=False,
        help='PMT gain model. Specify as (model_type, model_config, nT = True)')
```

We can look at the `gain_model_nv` config as an example. The "cmt://" section loads the "to_pe_model_nv" from CMT, `?version=ONLINE` queries the version and sets it to `ONLINE`, and `&run_id=plugin.run_id` substitues the run_id from the plugin at plugin.run_id.

How can we check which settings are hidden behind this cryptic URL?

The first option would be to simply initialize the plugin and check the corresponding option...

In [ ]:
p = st.get_single_plugin('020000', 'hitlets_nv')
p.config

... however this requires to know upfront how the corresponding parameter is called. <span style="color:red">In this case, if we look at the `setup()` method, `'gain_model_nv'` is loaded to `to_pe`</span>

In [ ]:
p.to_pe

Better to use the following:

In [ ]:
straxen.URLConfig.evaluate_dry("cmt://to_pe_model_nv?version=ONLINE&run_id=020000")

## Plugin Initialization

During the initialization of a plugin the `infer_dtype` and `setup` methods are executed first before processing the data. These methods are purely optional, and in fact one could specify the `dtype` directly.

The `infer_dtype` method is called first and helps to define the [dtype](https://numpy.org/doc/stable/user/basics.rec.html#structured-datatype-creation) of the structured array provided by the plugin. This is especially handy if the plugin returns multiple different data_kinds. 
```python
    def infer_dtype(self):
        return strax.hitlet_dtype()
```
E.g. the most simple dtype in strax is: 
```
[(('Start time since unix epoch [ns]', 'time'), numpy.int64),
 (('Exclusive end time since unix epoch [ns]', 'endtime'), numpy.int64)]
```

The setup method allows the user to define certain parameters which will always stay the same during the processing e.g. it allows to specify how many channels a detector has or which gains should be used: 
```python    
    def setup(self):
        self.channel_range = self.channel_map['nveto']
        self.n_channel = (self.channel_range[1] - self.channel_range[0]) + 1

        to_pe = self.gain_model_nv

        # Create to_pe array of size max channel:
        self.to_pe = np.zeros(self.channel_range[1] + 1, dtype=np.float32)
        self.to_pe[self.channel_range[0]:] = to_pe[:]

        # Check config of `hit_min_amplitude_nv` and define hit thresholds
        # if cmt config
        self.hit_thresholds = self.hit_min_amplitude_nv
```

## Compute & Optimized code:

Last but not least we have the `compute` method and your optimized code which is used to process/transform the information from one data_type into another.

```python
    def compute(self, hitlets_nv):    
        
        reduced_hitlets_nv = your_optimized_code(hitlets_nv, 
                                                 self.config['area_multiplier_nv']) 
        
        return reduced_hitlets_nv
    

def your_optimized_code(hitlets_nv, multiplier):
    rh = hitlets_nv
    rh['area'] * multiplier
    return rh
```

You can imagine the compute statement as a little script which is automatically applied to each chunk of the `data_type` the plugin depends on when making the new `provides` `data_type` (We will see in a minute an example). The return of the compute method is either an numpy.structured.array of the dtype you defined earlier or in case of a plugin with multiple outputs it is a dictionary with the `provides` names as key and arrays as arguments, e.g. like in [PulsePorcessing](https://github.com/XENONnT/straxen/blob/9a53370df918454ef7cec47be5ed685b97122568/straxen/plugins/records/records.py#L205-L207). 

Some comments about about the compute statement:
* In addition to the `depends_on` data_kind fields the compute method can also contain `start` and `end` as function arguments. `start` and `end` represent the `start` and `end` time of the corresponding chunk. This may be helpful if you want to compute a rate or make sure that your new data_type does not extend beyond the chunk boundaries.
* The arguments of the `compute` are named and sorted according to the `data_kinds` the plugin depends on. E.g. if you write a plugin like `straxen.EventBasics` which `depedens_on =  ('events', 'peak_basics', 'peak_positions', 'peak_proximity')` your compute method looks like `def compute(self, events, peaks):` If your plugin only `depedens_on =  ('peak_basics', 'peak_positions')` it will be just peaks etc..
* For some of our plugins we re-chunk our data after processing. This helps to reduce the overall file size of our data as the information quality gets more and more refined. In case your plugin depends on multiple differently chunked data_kinds strax will automatically find suitable gaps for your: 


![Example image strax chunking](images/MultipleChunks.png)

# How can I develop and test  plugins?

Now after we learned a lot about plugins in theory let us focus a bit more on handling and developing them. Usually a plugin evolves form a study you did in a notebook. You can imagine the `compute` method in a plugin as a notebook cell in which you did something like:

```python
data = st.get_array(run_id, 'data_kind', seconds_range=(0, 20))

# def compute(self, data_kind):
data_after_a = functionA(data)
data_after_b = functionB(data_after_a)
# return data_after_b
```
But instead of computing `data_after_b` only for the first 20 s (maybe for memory reasons) you would like to compute it for the whole run. Then your best choice is a plugin. In the following we will use the example plugin we have seen so far to check whether it is working or not.

First let us define the plugin:

In [ ]:
import strax
import straxen

import numpy as np

from immutabledict import immutabledict
export, __all__ = strax.exporter()

@export
class StandardPugin(strax.Plugin):
    """
    This is a default plugin, like used by many plugins in straxen. It 
    takes hitlets and multiplies their area with a user defined 
    multiplier.
    """
    __version__ = '0.0.1'
    
    depends_on = 'hitlets_nv'
    provides = 'new_hitlets_nv'
    data_kind = 'hitlets_nv'

    channel_map = straxen.URLConfig(track=False, type=immutabledict,
                                    help="immutabledict mapping subdetector to (min, max) "
                                         "channel number.")
    area_multiplier_nv = straxen.URLConfig(default=5, type=int,  track=True,
                                    help="Minimal hitlet area in pe.")

    parallel = True
    
    def infer_dtype(self):
        return strax.hitlet_dtype()
    
    def setup(self):
        start_ch, end_ch = self.channel_map['nveto']
        self.n_channel = end_ch - start_ch + 1
    
    def compute(self, hitlets_nv):    
        
        reduced_hitlets_nv = your_optimized_code(hitlets_nv, 
                                                 self.area_multiplier_nv) 
        
        return reduced_hitlets_nv
    

def your_optimized_code(hitlets_nv, multiplier):
    rh = np.copy(hitlets_nv)
    rh['area'] *= multiplier
    return rh

As usually we have to initialize our context first. Then we can register our plugin:

In [ ]:
st.register(StandardPugin)

This already worked now we can check the hash of the plugin....

In [ ]:
run_id = '025423'
st.key_for(run_id, 'new_hitlets_nv')

.... and its lineage:

In [ ]:
st.key_for(run_id, 'new_hitlets_nv').lineage

You should always make sure that the lineage shows all option you would like to track and does not show the option you do not want to track. 

So far everything seems to be fine. Before making the new data I would also recommend you to check your plugin first on a smaller scale and see if the output matches your expectations. For this purpose let us first initialize our plugin:

In [ ]:
plugin = st.get_single_plugin(run_id, 'new_hitlets_nv')

`plugin` is now the fully initialized plugin (`infer_dtype` and `setup`) already executed. Now we can use `st.get_iter` to iterate of chunks and data and see if the result matches our expectations (`st.get_iter` returns a so called [generator](https://www.programiz.com/python-programming/generator) sounds fancy but it simply means if you put it into a for loop you get one chunk at a time). 

In [ ]:
# st.get_iter(run_id, 'hitlets_nv', _chunk_number=1)
# Chunk number let you load a specific chunk
# this is especially helpful for debugging.
# Since it does not work I am now using break
for chunk in st.get_iter(run_id, 'hitlets_nv'):  
    hitlets = chunk.data                                            
    res = plugin.compute(hitlets)
    break

<div style="    
    color: black;
    background-color: #F6D092;
    border-style: solid;
    border-color: darkorange;
    border-radius : 5pt; 
    padding-left: 5pt" markdown="1">
    
# Exercise 1. Create your own little (regular) plugin: 

Now it is your turn write a new plugin. Write a plugin which computes the full width half maximum of the main and alternate S2 peak for each event. If there is no alternate peak, then set the FWHM is -1. Test your plugin on run `025423`. You can find some code to test your plugin in the hints, but try it first by yourself.


<details markdown="1">
<summary markdown="1">

**Hints (click me):**

</summary>

```python
# Now we can check if what we did was correct
st.register(S2FWHM)
pks = st.get_array('025423', 'peaks', _chunk_number = 0)

events = st.get_array('025423', 'event_basics')
sbc = strax.split_by_containment(pks, events)
main_s2_pks = np.array([peaks[event['s2_index']] for peaks, event in zip(sbc, events) if len(peaks)])
fullwidth_halfmax = st.get_array('025423', 's2_fwhm')

import matplotlib.pyplot as plt

for i in range(10):
    plt.plot(main_s2_pks['data'][i])
    plt.hlines(np.max(main_s2_pks['data'][i])*0.5, 0,200, linestyle = 'dashed')
    
    #raw wf
    first_over_half = np.where(main_s2_pks['data'][i]>np.max(main_s2_pks['data'][i])*0.5)[0][0]
    plt.hlines(np.max(main_s2_pks['data'][i])*0.5,
               first_over_half, first_over_half+fullwidth_halfmax['s2_fwhm'][i]/main_s2_pks['dt'][i], color = 'r')
    
    plt.show()
```
</details>

<details markdown="1">
<summary markdown="1">

**Solution (click me):**

</summary>

```python
import strax
import straxen
import numba
import numpy as np

from immutabledict import immutabledict
export, __all__ = strax.exporter()

@export
class S2FWHM(strax.Plugin):
    """
    This is a default plugin, like used by many plugins in straxen. It 
    finds the full-width half-maximum of the main and alternate S2
    peak for each event
    """
    __version__ = '0.0.1'
    
    depends_on = ('event_basics', 'peaks')
    provides = 's2_fwhm'
    data_kind = 'events'

    smoothing = straxen.URLConfig(default=False, type=bool, track = True,
                                    help="Flag for whether or not the"
                                         " waveform is smoothed or not.")
    averaging_samples = straxen.URLConfig(default=3, type=int,  track=True,
                                    help="Number of samples to average over.")
    
    # We really could just type the dtype, but for demonstration purposes
    # we can use infer_dtype
    def infer_dtype(self):
        #For the event times
        dtype = strax.time_fields
        for t in ['s2', 'alt_s2']:
            dtype += [((f'FWHM for {t} S2 in the event', f'{t}_fwhm'), np.float32)]

        return dtype
    
    def compute(self, events, peaks):    
        result = np.zeros(len(events), self.dtype)
        result['time'] = events['time']
        result['endtime'] = events['endtime']
        
        pks_per_ev = strax.split_by_containment(peaks, events)
        pk_buffer = {t: -999*np.ones((len(events),len(peaks['data'][0]))) for t in ['s2', 'alt_s2']}
        pk_dt = {t: 10*np.ones(len(events)) for t in ['s2', 'alt_s2']}
        
        for t in ['s2', 'alt_s2']:
            for i, (ev, pk_ev) in enumerate(zip(events, pks_per_ev)):
                pk_buffer[t][i] = pk_ev[ev[f'{t}_index']]['data']
                pk_dt[t][i] = pk_ev[ev[f'{t}_index']]['dt']
                
        
        for t in ['s2', 'alt_s2']:
            if self.smoothing:
                pk_buffer[t] = smooth(pk_buffer[t], self.averaging_samples)
            result[f'{t}_fwhm'] = fwhm(pk_buffer[t], pk_dt[t])
        
        return result

@numba.njit()
def fwhm(wfs, dts):
    result = -np.ones(len(wfs))
    for i, (w, dt) in enumerate(zip(wfs, dts)):
        if w[0] == -999:
            result[i] = -1
        else:
            above_half = np.where(w>=0.5*np.max(w))[0]
            result[i] = (above_half[-1]-above_half[0])*dt
    return result
    
@numba.njit()
def smooth(wfs, smooth_samps ):
    smooth_wfs = -999*np.ones((len(wfs),len(wfs[0])+smooth_samps-1))
    for wf_i, wf in enumerate(wfs):
        if wf[0]!=-999:
            smooth_wfs[wf_i] = np.convolve(wf, np.ones(smooth_samps)/smooth_samps)
                
    return smooth_wfs
```

</details>
</div>

In [ ]:
# Your code here